# Almaty Price Baseline v1

Этот ноутбук собирает один train-ready CSV в `datasets/processed/ads_model_v1.csv` и обучает baseline на `CatBoost` для предикта цены квартиры.

Текущая версия использует:
- базовые признаки объявления (`area`, `rooms`, `floor`, `year_built`, `district`, `house_type`, `condition`);
- гео-признаки по POI-датасетам;
- агрегаты по 5 ближайшим POI (`dist_1`, `mean_dist_3`, `mean_dist_5`, `count_500m`, `count_1000m`).

Осознанно не добавляю 5 ближайших объявлений из `ads.csv` в baseline v1, чтобы не занести leakage. Это логичный следующий шаг после текущего baseline.

In [2]:
from pathlib import Path
import sys

import pandas as pd

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
ROOT = next(
    candidate
    for candidate in ROOT_CANDIDATES
    if (candidate / 'src' / 'almaty_price_baseline.py').exists()
)
sys.path.insert(0, str(ROOT / 'src'))

from almaty_price_baseline import METRICS_PATH, PROCESSED_DATASET_PATH, run_pipeline

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

In [3]:
processed_df, metrics_df, model = run_pipeline()

print(f'Processed dataset: {PROCESSED_DATASET_PATH}')
print(f'Metrics file: {METRICS_PATH}')
print(f'Rows: {len(processed_df):,}')
print(f'Columns: {processed_df.shape[1]}')

metrics_df

Processed dataset: /Users/ruslan/Projects/ML-Project/datasets/processed/ads_model_v1.csv
Metrics file: /Users/ruslan/Projects/ML-Project/datasets/processed/baseline_metrics_v1.json
Rows: 39,301
Columns: 78


,model,mae_kzt,mape,rmse_kzt,rmse_log,r2
0,district_room_median,"12,404,500.18",0.18,"35,742,330.84",0.25,0.67
1,catboost_log_target,"5,774,477.78",0.09,"18,402,896.09",0.12,0.91


In [ ]:
base_columns = [
    'listing_id',
    'target_price_kzt',
    'area_m2',
    'rooms',
    'district',
    'house_type',
    'condition',
    'floor_current',
    'floors_total',
    'year_built',
    'metro_dist_km_1',
    'schools_count_1000m',
    'kindergartens_count_1000m',
    'hospitals_dist_km_1',
]

processed_df[base_columns].head(10)

In [ ]:
missing_share = (
    processed_df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename('missing_share')
)

missing_share.head(15).to_frame()

## Next Step

Если baseline v1 устраивает, следующий апгрейд стоит делать так:
1. считать признаки по 5 ближайшим объявлениям только на train-части;
2. добавлять `median_price_per_m2_5nn`, `mean_price_per_m2_5nn`, `mean_distance_5nn`;
3. валидировать отдельно random split и spatial split.